In [1]:
import pandas as pd
import numpy as np
import os
import glob
import warnings
warnings.filterwarnings('ignore')
from sqlalchemy import create_engine
import pymysql
import yfinance as yf
print("All libraries imported!")
print(pd.__version__)
print(yf.__version__)

All libraries imported!
2.2.3
1.4.0


In [4]:
from sqlalchemy import create_engine

DB_USER     = "root"
DB_PASSWORD = "admin123"
DB_HOST     = "127.0.0.1"
DB_PORT     = "3306"
DB_NAME     = "india_financial_stress"

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

with engine.connect() as conn:
    print("Connected to MySQL!")
    print(f"Database: {DB_NAME}")

Connected to MySQL!
Database: india_financial_stress


In [5]:
import os

BASE_PATH     = r"D:\workfiles\projects resume 1234567890\india_financial_stress"

SCREENER_PATH = os.path.join(BASE_PATH, "01_data", "screener")
NSE_PATH      = os.path.join(BASE_PATH, "01_data", "nse")
MACRO_PATH    = os.path.join(BASE_PATH, "01_data", "macro")

print(f"Screener folder: {os.path.exists(SCREENER_PATH)}")
print(f"NSE folder:      {os.path.exists(NSE_PATH)}")
print(f"Macro folder:    {os.path.exists(MACRO_PATH)}")

files = os.listdir(SCREENER_PATH)
print(f"Screener files:  {len(files)}")

Screener folder: True
NSE folder:      True
Macro folder:    True
Screener files:  50


In [11]:
import pandas as pd
import os

nse_file = os.path.join(NSE_PATH, "nifty100_companies.csv.csv")

# Read with proper column names
nse_df = pd.read_csv(nse_file, skiprows=1, header=None)

# Set column names manually
nse_df.columns = [
    'ticker', 'open', 'high', 'low', 
    'prev_close', 'current_price', 'indicative_close',
    'change', 'pct_change', 'volume', 'value',
    'high_52w', 'low_52w', 'change_30d', 'change_365d'
]

# Clean ticker column
nse_df['ticker'] = nse_df['ticker'].str.strip()

# Remove index row
nse_df = nse_df[nse_df['ticker'] != 'NIFTY 100']

# Clean current price
nse_df['current_price'] = nse_df['current_price'].astype(str)
nse_df['current_price'] = nse_df['current_price'].str.replace(',', '')
nse_df['current_price'] = pd.to_numeric(nse_df['current_price'], errors='coerce')

# Sector mapping
sector_map = {
    'HDFCBANK':'Banking','ICICIBANK':'Banking','SBIN':'Banking',
    'AXISBANK':'Banking','KOTAKBANK':'Banking','INDUSINDBK':'Banking',
    'BANKBARODA':'Banking','CANBK':'Banking','FEDERALBNK':'Banking',
    'IDFCFIRSTB':'Banking',
    'INFY':'IT','TCS':'IT','WIPRO':'IT','HCLTECH':'IT','TECHM':'IT',
    'MPHASIS':'IT','COFORGE':'IT','PERSISTENT':'IT','OFSS':'IT',
    'KPITTECH':'IT',
    'RELIANCE':'Energy','ONGC':'Energy','NTPC':'Energy',
    'POWERGRID':'Energy','COALINDIA':'Energy','BPCL':'Energy',
    'IOC':'Energy','GAIL':'Energy','TATAPOWER':'Energy',
    'ADANIGREEN':'Energy',
    'HINDUNILVR':'FMCG','ITC':'FMCG','NESTLEIND':'FMCG',
    'BRITANNIA':'FMCG','DABUR':'FMCG','MARICO':'FMCG',
    'COLPAL':'FMCG','GODREJCP':'FMCG','TATACONSUM':'FMCG',
    'EMAMILTD':'FMCG',
    'SUNPHARMA':'Pharma','DRREDDY':'Pharma','CIPLA':'Pharma',
    'DIVISLAB':'Pharma','LUPIN':'Pharma','AUROPHARMA':'Pharma',
    'TORNTPHARM':'Pharma','ALKEM':'Pharma','IPCA':'Pharma',
    'GLENMARK':'Pharma'
}

# Filter our 50 companies
nse_df = nse_df[nse_df['ticker'].isin(sector_map.keys())]

# Add columns
nse_df['sector']       = nse_df['ticker'].map(sector_map)
nse_df['company_name'] = nse_df['ticker']
nse_df['nse_symbol']   = nse_df['ticker']

# Keep only needed columns
nse_df = nse_df[['ticker','company_name','sector','nse_symbol','current_price']]

print(f"Companies found: {len(nse_df)}")
print(nse_df.head(10))

Companies found: 33
       ticker company_name   sector nse_symbol  current_price
1    HDFCBANK     HDFCBANK  Banking   HDFCBANK         769.00
2   ICICIBANK    ICICIBANK  Banking  ICICIBANK        1267.20
4        INFY         INFY       IT       INFY        1175.00
6    RELIANCE     RELIANCE   Energy   RELIANCE        1358.00
8         ITC          ITC     FMCG        ITC         302.05
9       WIPRO        WIPRO       IT      WIPRO         202.97
11       SBIN         SBIN  Banking       SBIN         949.60
12  SUNPHARMA    SUNPHARMA   Pharma  SUNPHARMA        1840.00
13   AXISBANK     AXISBANK  Banking   AXISBANK        1287.00
14        TCS          TCS       IT        TCS        2317.90


In [12]:
all_50 = set(sector_map.keys())
found  = set(nse_df['ticker'].tolist())
missing = all_50 - found

print(f"Missing {len(missing)} companies:")
for m in sorted(missing):
    print(f"  --> {m}")

Missing 17 companies:
  --> ALKEM
  --> AUROPHARMA
  --> COFORGE
  --> COLPAL
  --> DABUR
  --> EMAMILTD
  --> FEDERALBNK
  --> GLENMARK
  --> IDFCFIRSTB
  --> INDUSINDBK
  --> IPCA
  --> KPITTECH
  --> LUPIN
  --> MARICO
  --> MPHASIS
  --> OFSS
  --> PERSISTENT


In [13]:
missing_companies = [
    {'ticker': 'ALKEM',      'current_price': 5200.00,  'sector': 'Pharma'},
    {'ticker': 'AUROPHARMA', 'current_price': 1180.00,  'sector': 'Pharma'},
    {'ticker': 'COFORGE',    'current_price': 7800.00,  'sector': 'IT'},
    {'ticker': 'COLPAL',     'current_price': 2800.00,  'sector': 'FMCG'},
    {'ticker': 'DABUR',      'current_price': 480.00,   'sector': 'FMCG'},
    {'ticker': 'EMAMILTD',   'current_price': 420.00,   'sector': 'FMCG'},
    {'ticker': 'FEDERALBNK', 'current_price': 185.00,   'sector': 'Banking'},
    {'ticker': 'GLENMARK',   'current_price': 1050.00,  'sector': 'Pharma'},
    {'ticker': 'IDFCFIRSTB', 'current_price': 62.00,    'sector': 'Banking'},
    {'ticker': 'INDUSINDBK', 'current_price': 820.00,   'sector': 'Banking'},
    {'ticker': 'IPCA',       'current_price': 1480.00,  'sector': 'Pharma'},
    {'ticker': 'KPITTECH',   'current_price': 1380.00,  'sector': 'IT'},
    {'ticker': 'LUPIN',      'current_price': 2100.00,  'sector': 'Pharma'},
    {'ticker': 'MARICO',     'current_price': 580.00,   'sector': 'FMCG'},
    {'ticker': 'MPHASIS',    'current_price': 2900.00,  'sector': 'IT'},
    {'ticker': 'OFSS',       'current_price': 9800.00,  'sector': 'IT'},
    {'ticker': 'PERSISTENT', 'current_price': 5100.00,  'sector': 'IT'},
]

# Convert to dataframe
missing_df = pd.DataFrame(missing_companies)
missing_df['company_name'] = missing_df['ticker']
missing_df['nse_symbol']   = missing_df['ticker']

# Combine with existing 33
nse_df = pd.concat([nse_df, missing_df], ignore_index=True)

print(f"Total companies: {len(nse_df)}")
print(nse_df.groupby('sector').count()['ticker'])

Total companies: 50
sector
Banking    10
Energy     10
FMCG       10
IT         10
Pharma     10
Name: ticker, dtype: int64


In [14]:
# Cell 6 — Load companies into MySQL
nse_df.to_sql(
    name='companies',
    con=engine,
    if_exists='append',
    index=False
)

print("Companies loaded into MySQL!")
print(f"Total rows inserted: {len(nse_df)}")


Companies loaded into MySQL!
Total rows inserted: 50


In [16]:
import pandas as pd
import os

macro_file = os.path.join(MACRO_PATH, "rbi_macro_data.csv")
macro_df = pd.read_csv(macro_file)

# Split period_key into year and month
macro_df['macro_year']  = macro_df['year_month'].str[:4].astype(int)
macro_df['macro_month'] = macro_df['year_month'].str[5:7].astype(int)

# Rename columns to match MySQL table
macro_df = macro_df.rename(columns={
    'year_month':          'period_key',
    'reverse_repo_rate':   'rev_repo_rate',
    'gdp_growth_quarterly':'gdp_growth'
})

# Load into MySQL
macro_df.to_sql(
    name='macro_indicators',
    con=engine,
    if_exists='append',
    index=False
)

print("Macro data loaded into MySQL!")
print(f"Total rows inserted: {len(macro_df)}")
print(macro_df.head(3))

Macro data loaded into MySQL!
Total rows inserted: 129
  period_key  repo_rate  rev_repo_rate  cpi_inflation  gdp_growth  macro_year  \
0    2015-04       7.50           6.50           4.87         7.5        2015   
1    2015-05       7.25           6.25           5.01         7.5        2015   
2    2015-06       7.25           6.25           5.40         7.5        2015   

   macro_month  
0            4  
1            5  
2            6  


In [18]:
# Cell 8 — Load all 50 Screener files
import glob

screener_files = glob.glob(os.path.join(SCREENER_PATH, "*.xlsx"))
print(f"Found {len(screener_files)} files")

all_data = []

for filepath in screener_files:
    try:
        ticker = os.path.basename(filepath).replace('.xlsx','').upper().strip()
        wb_df  = pd.read_excel(filepath, sheet_name='Data Sheet',
                               header=None, engine='openpyxl')

        years_row = wb_df.iloc[15, 1:11]
        years     = pd.to_datetime(years_row, errors='coerce').dt.year.tolist()

        def get_row(idx):
            return wb_df.iloc[idx, 1:11].tolist()

        revenue           = get_row(16)
        other_income      = get_row(24)
        employee_cost     = get_row(21)
        other_expenses    = get_row(23)
        depreciation      = get_row(25)
        interest_expense  = get_row(26)
        profit_before_tax = get_row(27)
        tax_amount        = get_row(28)
        net_profit        = get_row(29)
        equity_capital    = get_row(56)
        reserves          = get_row(57)
        total_borrowings  = get_row(58)
        other_liabilities = get_row(59)
        total_assets      = get_row(60)
        net_block         = get_row(61)
        investments       = get_row(63)
        cash_and_bank     = get_row(68)
        num_equity_shares = get_row(69)
        cfo               = get_row(81)
        cfi               = get_row(82)
        cff               = get_row(83)
        net_cash_flow     = get_row(84)

        for i, yr in enumerate(years):
            try:
                yr = int(yr)
            except:
                continue
            if yr < 2015:
                continue
            all_data.append({
                'ticker':             ticker,
                'report_year':        yr,
                'revenue':            revenue[i],
                'other_income':       other_income[i],
                'employee_cost':      employee_cost[i],
                'other_expenses':     other_expenses[i],
                'depreciation':       depreciation[i],
                'interest_expense':   interest_expense[i],
                'profit_before_tax':  profit_before_tax[i],
                'tax_amount':         tax_amount[i],
                'net_profit':         net_profit[i],
                'equity_capital':     equity_capital[i],
                'reserves':           reserves[i],
                'total_borrowings':   total_borrowings[i],
                'other_liabilities':  other_liabilities[i],
                'total_assets':       total_assets[i],
                'net_block':          net_block[i],
                'investments':        investments[i],
                'cash_and_bank':      cash_and_bank[i],
                'num_equity_shares':  num_equity_shares[i],
                'cfo':                cfo[i],
                'cfi':                cfi[i],
                'cff':                cff[i],
                'net_cash_flow':      net_cash_flow[i],
            })
        print(f"✅ {ticker} — done")

    except Exception as e:
        print(f"❌ {ticker} — Error: {e}")

print(f"\nTotal records collected: {len(all_data)}")

Found 50 files
✅ ADANI GREEN — done
✅ ALKEM LAB — done
✅ AUROBINDO PHARMA — done
✅ AXIS BANK — done
✅ B P C L — done
✅ BANK OF BARODA — done
✅ BRITANNIA INDS — done
✅ CANARA BANK — done
✅ CIPLA — done
✅ COAL INDIA — done
✅ COFORGE — done
✅ COLGATE-PALMOLIV — done
✅ DABUR INDIA — done
✅ DIVI'S LAB — done
✅ DR REDDY'S LABS — done
✅ EMAMI — done
✅ FEDERAL BANK — done
✅ GAIL (INDIA) — done
✅ GLENMARK PHARMA — done
✅ GODREJ CONSUMER — done
✅ HCL TECHNOLOGIES — done
✅ HDFC BANK — done
✅ HIND. UNILEVER — done
✅ I O C L — done
✅ ICICI BANK — done
✅ IDFC FIRST BANK — done
✅ INDUSIND BANK — done
✅ INFOSYS — done
✅ IPCA LABS — done
✅ ITC — done
✅ KOTAK MAH. BANK — done
✅ KPIT TECHNOLOGI — done
✅ LUPIN — done
✅ MARICO — done
✅ MPHASIS — done
✅ NESTLE INDIA — done
✅ NTPC — done
✅ O N G C — done
✅ ORACLE FIN.SERV — done
✅ PERSISTENT SYSTEMS — done
✅ POWER GRID CORPN — done
✅ RELIANCE INDUSTRIES — done
✅ SBIN — done
✅ SUN PHARMA.INDS — done
✅ TATA CONSUMER — done
✅ TATA POWER CO — done
✅ TCS — done
✅

In [19]:
# Cell 9 — Save to MySQL
fin_df = pd.DataFrame(all_data)

# Convert all numeric columns
numeric_cols = [
    'revenue','other_income','employee_cost','other_expenses',
    'depreciation','interest_expense','profit_before_tax',
    'tax_amount','net_profit','equity_capital','reserves',
    'total_borrowings','other_liabilities','total_assets',
    'net_block','investments','cash_and_bank','num_equity_shares',
    'cfo','cfi','cff','net_cash_flow'
]

for col in numeric_cols:
    fin_df[col] = pd.to_numeric(fin_df[col], errors='coerce')

print(f"Total records: {len(fin_df)}")
print(f"Years covered: {sorted(fin_df['report_year'].unique())}")
print(f"Companies: {fin_df['ticker'].nunique()}")
print(fin_df.head(3))

Total records: 499
Years covered: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]
Companies: 50
        ticker  report_year  revenue  other_income  employee_cost  \
0  ADANI GREEN         2017   501.65         80.30          39.05   
1  ADANI GREEN         2018  1480.28         41.51          43.71   
2  ADANI GREEN         2019  2057.98         72.74          59.69   

   other_expenses  depreciation  interest_expense  profit_before_tax  \
0            2.92        333.27            334.14            -183.92   
1           11.08        542.99            551.82            -210.18   
2          199.82       1061.96           1121.18            -584.66   

   tax_amount  ...  other_liabilities  total_assets  net_block  investments  \
0     -137.68  ...             610.41       6159.92    4341.14        26.47   
1      -72.70  ...            4503.04      15708.58

In [21]:
# Cell 10 — Fix ticker names then load to MySQL

# Map full names from screener to NSE ticker symbols
ticker_map = {
    'ADANI GREEN':          'ADANIGREEN',
    'ALKEM LAB':            'ALKEM',
    'AUROBINDO PHARMA':     'AUROPHARMA',
    'AXIS BANK':            'AXISBANK',
    'B P C L':              'BPCL',
    'BANK OF BARODA':       'BANKBARODA',
    'BRITANNIA INDS':       'BRITANNIA',
    'CANARA BANK':          'CANBK',
    'CIPLA':                'CIPLA',
    'COAL INDIA':           'COALINDIA',
    'COFORGE':              'COFORGE',
    'COLGATE-PALMOLIV':     'COLPAL',
    'DABUR INDIA':          'DABUR',
    "DIVI'S LAB":           'DIVISLAB',
    "DR REDDY'S LABS":      'DRREDDY',
    'EMAMI':                'EMAMILTD',
    'FEDERAL BANK':         'FEDERALBNK',
    'GAIL (INDIA)':         'GAIL',
    'GLENMARK PHARMA':      'GLENMARK',
    'GODREJ CONSUMER':      'GODREJCP',
    'HCL TECHNOLOGIES':     'HCLTECH',
    'HDFC BANK':            'HDFCBANK',
    'HIND. UNILEVER':       'HINDUNILVR',
    'I O C L':              'IOC',
    'ICICI BANK':           'ICICIBANK',
    'IDFC FIRST BANK':      'IDFCFIRSTB',
    'INDUSIND BANK':        'INDUSINDBK',
    'INFOSYS':              'INFY',
    'IPCA LABS':            'IPCA',
    'ITC':                  'ITC',
    'KOTAK MAH. BANK':      'KOTAKBANK',
    'KPIT TECHNOLOGI':      'KPITTECH',
    'LUPIN':                'LUPIN',
    'MARICO':               'MARICO',
    'MPHASIS':              'MPHASIS',
    'NESTLE INDIA':         'NESTLEIND',
    'NTPC':                 'NTPC',
    'O N G C':              'ONGC',
    'ORACLE FIN.SERV':      'OFSS',
    'PERSISTENT SYSTEMS':   'PERSISTENT',
    'POWER GRID CORPN':     'POWERGRID',
    'RELIANCE INDUSTRIES':  'RELIANCE',
    'SBIN':                 'SBIN',
    'SUN PHARMA.INDS':      'SUNPHARMA',
    'TATA CONSUMER':        'TATACONSUM',
    'TATA POWER CO':        'TATAPOWER',
    'TCS':                  'TCS',
    'TECH MAHINDRA':        'TECHM',
    'TORRENT PHARMA':       'TORNTPHARM',
    'WIPRO':                'WIPRO',
}

# Apply mapping
fin_df['ticker'] = fin_df['ticker'].map(ticker_map)

# Check any unmapped
unmapped = fin_df[fin_df['ticker'].isna()]
print(f"Unmapped tickers: {len(unmapped)}")
print(f"Unique tickers: {fin_df['ticker'].nunique()}")

# Load to MySQL
fin_df.to_sql(
    name='financial_data',
    con=engine,
    if_exists='append',
    index=False
)

print(f"Financial data loaded! Total rows: {len(fin_df)}")

Unmapped tickers: 0
Unique tickers: 50
Financial data loaded! Total rows: 499


In [26]:
# Cell 11 — Fetch stock prices fixed for new yfinance
import yfinance as yf

tickers_nse = [t + ".NS" for t in ticker_map.values()]
print(f"Fetching prices for {len(tickers_nse)} companies...")

all_prices = []

for nse_ticker in tickers_nse:
    try:
        ticker_clean = nse_ticker.replace(".NS","")
        
        data = yf.download(
            nse_ticker,
            start="2015-01-01",
            end="2026-05-01",
            progress=False
        )
        
        if data.empty:
            print(f"❌ {ticker_clean} — no data")
            continue

        # Flatten multi-level columns
        data.columns = [col[0] for col in data.columns]
        
        # Date is in index — reset it
        data = data.reset_index()
        data = data.rename(columns={'index': 'Date'})
        
        # Convert date
        data['Date'] = pd.to_datetime(data['Date']).dt.date

        for _, row in data.iterrows():
            try:
                all_prices.append({
                    'ticker':      ticker_clean,
                    'price_date':  row['Date'],
                    'open_price':  round(float(row['Open']), 2),
                    'high_price':  round(float(row['High']), 2),
                    'low_price':   round(float(row['Low']), 2),
                    'close_price': round(float(row['Close']), 2),
                    'vol':         int(row['Volume']),
                })
            except:
                continue

        print(f"✅ {ticker_clean} — {len(data)} days")

    except Exception as e:
        print(f"❌ {ticker_clean} — Error: {e}")

print(f"\nTotal price records: {len(all_prices)}")

Fetching prices for 50 companies...
✅ ADANIGREEN — 1942 days
✅ ALKEM — 2555 days
✅ AUROPHARMA — 2795 days
✅ AXISBANK — 2796 days
✅ BPCL — 2796 days
✅ BANKBARODA — 2796 days
✅ BRITANNIA — 2796 days
✅ CANBK — 2796 days
✅ CIPLA — 2796 days
✅ COALINDIA — 2796 days
✅ COFORGE — 2796 days
✅ COLPAL — 2796 days
✅ DABUR — 2796 days
✅ DIVISLAB — 2796 days
✅ DRREDDY — 2796 days
✅ EMAMILTD — 2796 days
✅ FEDERALBNK — 2796 days
✅ GAIL — 2796 days
✅ GLENMARK — 2796 days
✅ GODREJCP — 2796 days
✅ HCLTECH — 2796 days
✅ HDFCBANK — 2796 days
✅ HINDUNILVR — 2796 days
✅ IOC — 2796 days
✅ ICICIBANK — 2796 days
✅ IDFCFIRSTB — 2586 days
✅ INDUSINDBK — 2796 days


$IPCA.NS: possibly delisted; no price data found  (1d 2015-01-01 -> 2026-05-01)

1 Failed download:
['IPCA.NS']: possibly delisted; no price data found  (1d 2015-01-01 -> 2026-05-01)


✅ INFY — 2796 days
❌ IPCA — no data
✅ ITC — 2796 days
✅ KOTAKBANK — 2796 days
✅ KPITTECH — 1737 days
✅ LUPIN — 2796 days
✅ MARICO — 2796 days
✅ MPHASIS — 2796 days
✅ NESTLEIND — 2796 days
✅ NTPC — 2796 days
✅ ONGC — 2796 days
✅ OFSS — 2796 days
✅ PERSISTENT — 2796 days
✅ POWERGRID — 2796 days
✅ RELIANCE — 2796 days
✅ SBIN — 2796 days
✅ SUNPHARMA — 2796 days
✅ TATACONSUM — 2796 days
✅ TATAPOWER — 2796 days
✅ TCS — 2796 days
✅ TECHM — 2796 days
✅ TORNTPHARM — 2796 days
✅ WIPRO — 2796 days

Total price records: 134639


In [27]:
# Cell 12 — Save stock prices to MySQL
prices_df = pd.DataFrame(all_prices)

print(f"Total records: {len(prices_df)}")
print(f"Companies: {prices_df['ticker'].nunique()}")
print(f"Date range: {prices_df['price_date'].min()} to {prices_df['price_date'].max()}")

prices_df.to_sql(
    name='stock_prices',
    con=engine,
    if_exists='append',
    index=False,
    chunksize=1000
)

print("Stock prices loaded into MySQL!")

Total records: 134639
Companies: 49
Date range: 2015-01-01 to 2026-04-30
Stock prices loaded into MySQL!
